# FutureWork AI — All-Occupation Career Navigator (VSA Edition)

**910 O*NET occupations** receive skill-based recommendations. All 910 also receive *candidate* Australian occupation crosswalk suggestions, generated automatically from job-title similarity. The Victorian Skills Authority (VSA) workforce projections are used **only through Australian ANZSCO-4 occupation groups**.

**Disclosure:** Only manually reviewed mappings are treated as reviewed. Automated mappings are *provisional*, are not official equivalences, and their workforce statistics are exploratory. A 2025–2028 new-workers forecast includes growth and replacement demand; it is not the number of vacancies. This tool does not predict the probability of employment.

## 1 — Load the prepared Phase 1 data
Keep this notebook next to Phases 1–3, and check `processed/occupation_skill_matrix.csv` plus `processed/occupation_lookup.csv` exist.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity

ROOT=Path.cwd(); PROCESSED=ROOT/'processed'; DATA=ROOT/'data'; RESULTS=ROOT/'results'
DATA.mkdir(exist_ok=True); RESULTS.mkdir(exist_ok=True)
for file in ('occupation_skill_matrix.csv','occupation_lookup.csv'):
    if not (PROCESSED/file).exists(): raise FileNotFoundError(f'Missing {PROCESSED/file}. Run Phase 1 first.')
raw=pd.read_csv(PROCESSED/'occupation_skill_matrix.csv',dtype={'occupation_code':str})
lookup=pd.read_csv(PROCESSED/'occupation_lookup.csv',dtype={'occupation_code':str}).drop_duplicates('occupation_code').set_index('occupation_code')
X=raw.set_index('occupation_code').apply(pd.to_numeric,errors='coerce')
X=X.loc[X.index.intersection(lookup.index)]
coverage=X.notna().mean(axis=1); eligible=coverage[coverage>=0.60].index
filled=X.loc[eligible].fillna(X.loc[eligible].median()).fillna(1)
SIM=pd.DataFrame(cosine_similarity(np.maximum(filled.to_numpy()-1,0)),index=eligible,columns=eligible)
print(f'Loaded {len(eligible)} eligible occupations / {len(X.columns)} skills')

## 2 — Mappings and Victorian projections
Verified prototype occupation mappings are distinguished from **provisional exact-title matches**, which have not undergone duty-by-duty validation. Victorian government numbers are ANZSCO4 group projections, not vacancies for a six-digit occupation.


In [ ]:
mapfile=DATA/'verified_mapping.csv'; marketfile=DATA/'au_market_signals.csv'
if not mapfile.exists():
    pd.DataFrame(columns=['occupation_code','au_code','au_classification','reviewed','source_note']).to_csv(mapfile,index=False)
if not marketfile.exists():
    pd.DataFrame(columns=['au_code','au_classification','growth_pct_10yr','shortage_status','source_note']).to_csv(marketfile,index=False)
print('Mapping template:',mapfile,'| Labour-market template:',marketfile)
print('Fill with VERIFIED data if you want real demand-aware rankings. Never treat blank values as low demand.')

## 3 — Read and validate market signals
When no verified data are supplied, results remain **skills-only** and demand is labelled **unknown**. We intentionally do *not* insert fictional growth/shortage figures.

In [ ]:
# Read reviewed maps and JSA evidence; VSA CSV is BUNDLED in the data folder.
M=pd.read_csv(mapfile,dtype=str).fillna(''); A=pd.read_csv(marketfile,dtype=str).fillna('')
for frame in (M,A):
    frame['au_code']=frame.au_code.astype(str).str.strip()
    frame['au_classification']=frame.au_classification.astype(str).str.strip().str.upper()
M['occupation_code']=M.occupation_code.astype(str).str.strip()
M=M[(M.reviewed.str.lower().str.strip()=='yes') & M.occupation_code.isin(eligible) & M.au_code.str.fullmatch(r'\d{6}')].drop_duplicates('occupation_code').copy()
A=A[A.au_code.str.fullmatch(r'\d{6}')].drop_duplicates(['au_code','au_classification']).copy()
A['growth_pct_10yr']=pd.to_numeric(A.growth_pct_10yr,errors='coerce')
market=M.merge(A,on=['au_code','au_classification'],how='left',suffixes=('_mapping','_market')).set_index('occupation_code')

vsa_path=DATA/'vic_anzsco4_workforce.csv'
if not vsa_path.exists():
    raise FileNotFoundError('Missing VSA file: '+str(vsa_path)+' — upload the data/ directory from the new ZIP, not just the notebook.')
VIC=pd.read_csv(vsa_path,dtype={'anzsco4':str})
VIC['anzsco4']=VIC.anzsco4.astype(str).str.zfill(4)
required=['region','anzsco4','employment_2025','employment_growth_2025_2035','new_workers_2025_2028','new_workers_2025_2035']
assert set(required).issubset(VIC.columns), 'Wrong VSA CSV structure'
for col in required[2:]:VIC[col]=pd.to_numeric(VIC[col],errors='coerce')
print('VSA loaded:',len(VIC),'rows,',VIC.region.nunique(),'regions,',VIC.anzsco4.nunique(),'occupation groups')
print('Previously reviewed mappings:',len(market))

# Candidate match EVERY O*NET occupation against official Australian occupation titles.
# These are algorithmic proposals, NOT verified crosswalks.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as text_cosine
au_titles=pd.read_csv(DATA/'au_occupation_titles.csv',dtype={'au_code':str}).dropna(subset=['au_code','au_title'])
au_titles=au_titles.drop_duplicates('au_code').reset_index(drop=True)
source_list=lookup.loc[eligible,['occupation_title']].reset_index()
assert len(source_list)==len(eligible)
vectorizer=TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),lowercase=True)
T=vectorizer.fit_transform(source_list.occupation_title.astype(str).tolist()+au_titles.au_title.astype(str).tolist())
raw_match=text_cosine(T[:len(source_list)],T[len(source_list):])
import numpy as np
best=np.argsort(-raw_match,axis=1)[:,:3]
rows=[]
for i,src in source_list.iterrows():
    for rank,j in enumerate(best[i],1):
        au=au_titles.iloc[int(j)]
        rows.append({'occupation_code':str(src.occupation_code),'occupation_title':src.occupation_title,'candidate_rank':rank,
                     'au_code':str(au.au_code),'au_title':str(au.au_title),'title_similarity':round(float(raw_match[i,j]),5),
                     'mapping_status':'UNVERIFIED_AUTOMATIC'})
crosswalk=pd.DataFrame(rows)
# Preserve manually reviewed mappings as authoritative, even if title score ranks another candidate higher.
reviewed_codes=dict(zip(M.occupation_code,M.au_code))
for code,au_code in reviewed_codes.items():
    crosswalk.loc[(crosswalk.occupation_code==code)&(crosswalk.au_code==str(au_code)),'mapping_status']='REVIEWED_PROTOTYPE'
CROSSWALK=crosswalk
CROSSWALK.to_csv(RESULTS/'all_910_candidate_mappings.csv',index=False)
print('Candidate coverage:',CROSSWALK.occupation_code.nunique(),'of',len(eligible),'occupations; 3 candidates each')
print('Candidates are NOT government-verified occupation mappings.')


## 4 — Relevant opportunity evidence only
Re-ranking is limited to the 50 closest skill-related occupations. Where mapped evidence is missing, results explicitly say so. Title matches remain provisional.


In [ ]:
# Region-specific VSA evidence is attached to top skill-neighbour recommendations.
# Tentative AU evidence is displayed as indicative but NEVER marked verified.
def demand_recommend(source_code,top_n=5,region='Victoria',candidate_pool=50):
    source_code=str(source_code)
    if source_code not in SIM.index: raise ValueError('Unknown occupation')
    scores=SIM.loc[source_code].drop(source_code).sort_values(ascending=False).rename('similarity').reset_index().rename(columns={'index':'occupation_code'})
    scores['occupation_title']=scores.occupation_code.map(lookup.occupation_title)
    best=CROSSWALK[CROSSWALK.candidate_rank==1].set_index('occupation_code')
    scores['au_code']=scores.occupation_code.map(best.au_code).fillna('')
    scores['au_title']=scores.occupation_code.map(best.au_title).fillna('')
    scores['title_confidence']=scores.occupation_code.map(best.title_similarity)
    scores['mapping_quality']='unverified_title_candidate'
    for code,au in reviewed_codes.items():
        mask=scores.occupation_code==code
        scores.loc[mask,'au_code']=str(au)
        scores.loc[mask,'mapping_quality']='reviewed_prototype'
    scores['anzsco4']=scores.au_code.astype(str).str[:4]
    region_values=VIC if region=='Victoria' else VIC[VIC.region==region]
    grouped=region_values.groupby('anzsco4',as_index=False)[['employment_2025','employment_growth_2025_2035','new_workers_2025_2028','new_workers_2025_2035']].sum(min_count=1)
    grouped['growth_pct_2025_2035']=np.where(grouped.employment_2025>0,100*grouped.employment_growth_2025_2035/grouped.employment_2025,np.nan)
    scores=scores.merge(grouped,on='anzsco4',how='left',validate='many_to_one')
    scores['evidence_status']=np.where(scores.mapping_quality=='reviewed_prototype','VSA_GROUP_REVIEWED_MAPPING','VSA_GROUP_UNVERIFIED_MAPPING')
    top=scores.head(top_n).copy()
    pool=scores.head(candidate_pool).copy()
    # For the primary opportunity recommendation view, keep ONLY strong title candidates or reviewed mappings;
    # do not promote unrelated job titles with low similarity.
    minimum_title_confidence=0.72
    accepted=pool[(pool.mapping_quality=='reviewed_prototype') | (pool.title_confidence>=minimum_title_confidence)].copy()
    accepted=accepted[accepted.growth_pct_2025_2035.notna()].copy()
    if accepted.empty:
        status=f'No sufficiently supported AU matches in the {candidate_pool} closest skill occupations. Showing skills-only options (no invented demand).'
        return top,None,status,scores
    # Growth-percentile is calculated only across eligible candidates for this source; results are exploratory.
    accepted['growth_rank']=accepted.growth_pct_2025_2035.rank(pct=True)
    accepted['combined_score']=.7*accepted.similarity+.3*accepted.growth_rank
    accepted=accepted.sort_values('combined_score',ascending=False).head(top_n)
    provisional=(accepted.mapping_quality!='reviewed_prototype').sum()
    status=(f'{len(accepted)} market-linked candidates among {candidate_pool} close skill matches; '
            f'{provisional} rely on UNVERIFIED title mappings. VSA projections are for ANZSCO4 GROUPS, not individual jobs.')
    return top,accepted,status,scores

def search_jobs(text):
    return lookup.loc[eligible][lookup.loc[eligible,'occupation_title'].str.contains(text,case=False,na=False)].head(12)[['occupation_title']]


## 5 — Model self-test
This default test selects an occupation for reproducibility; use the interactive dropdown for any of the 910 occupations.


In [ ]:
source_code = eligible[0]
skill_top, opportunity, status, all_candidates = demand_recommend(source_code)
print(f'Ready: {len(eligible)} selectable occupations and {len(M)} reviewed market mappings.')
print('Default smoke-test source:', lookup.loc[source_code,'occupation_title'])
assert len(skill_top)==5

## 6 — Interactive navigator for all eligible occupations
**Occupation search** filters the full list; choose any occupation. The system computes its top-five skills matches from all other eligible roles. **Target comparison** explains shared skills, differences in occupational importance, and potential learning priorities based on *optional* self-rated confidence. Market ranking is shown **only** when sufficient reviewed mappings exist. Neither 1–5 confidence scores nor group-level employment growth are hiring probabilities.

In [ ]:
from IPython.display import clear_output

def compare_skills(source, target, confidence=None, n=10):
    if source not in X.index or target not in X.index:
        raise ValueError('Occupation not available')
    a=X.loc[source]; b=X.loc[target]
    pair=pd.DataFrame({'skill_code':X.columns,'current_importance':a.values,'target_importance':b.values})
    pair=pair.dropna(subset=['current_importance','target_importance']).copy()
    pair['difference']=pair.target_importance-pair.current_importance
    if (PROCESSED/'skill_dictionary.csv').exists():
        dictionary=pd.read_csv(PROCESSED/'skill_dictionary.csv',dtype=str)
        print('Skill dictionary columns:',list(dictionary.columns)) if False else None
        key=next((c for c in ('element_id','skill_code','skill_id') if c in dictionary.columns),None)
        val=next((c for c in ('skill_name','element_name','name') if c in dictionary.columns),None)
        if key and val:
            labels=dict(zip(dictionary[key],dictionary[val]));pair['skill']=pair.skill_code.map(labels).fillna(pair.skill_code)
        else: pair['skill']=pair.skill_code
    else: pair['skill']=pair.skill_code
    shared=pair.sort_values(['current_importance','target_importance'],ascending=False).head(n)
    gaps=pair[pair.difference>0].sort_values('difference',ascending=False).head(n)
    return shared,gaps,pair

def show_career_report(source,target=None,confidence=None,save=False,region="Victoria"):
    top,market_rank,market_status,all_scores=demand_recommend(source,region=region)
    display(Markdown('### Career options for **'+str(lookup.loc[source,'occupation_title'])+'**'))
    display(top[['occupation_title','similarity','growth_pct_2025_2035','new_workers_2025_2028','evidence_status']].reset_index(drop=True))
    display(Markdown('**Victorian occupation-group projections, not individual job vacancies.** '+market_status))
    if market_rank is not None:
        display(market_rank[['occupation_title','similarity','growth_pct_2025_2035','new_workers_2025_2028','mapping_quality','combined_score']].reset_index(drop=True))
    else: print('For this occupation, market-ranked recommendations are not available yet. Skills-only recommendations remain available.')
    if target is None:target=top.iloc[0].occupation_code
    shared,gaps,pair=compare_skills(source,target)
    display(Markdown('### Skill comparison: '+str(lookup.loc[source,'occupation_title'])+' → '+str(lookup.loc[target,'occupation_title'])))
    display(Markdown('**Common important skills (occupation data):**'))
    display(shared[['skill','current_importance','target_importance']].reset_index(drop=True))
    display(Markdown('**Skills with higher importance in the target occupation:**'))
    display(gaps[['skill','current_importance','target_importance','difference']].reset_index(drop=True))
    if confidence:
        priority=pair[pair.skill_code.isin(confidence)].copy()
        priority['self_confidence']=priority.skill_code.map(confidence)
        priority=priority[(priority.self_confidence<=2)&(priority.target_importance>=3)].sort_values('target_importance',ascending=False)
        display(Markdown('**Potential learning priorities (self-ratings ≤2/5, target importance ≥3):**'))
        if len(priority):display(priority[['skill','target_importance','self_confidence']].head(8).reset_index(drop=True))
        else:print('None flagged among the assessed skills. This does not certify competence.')
    display(Markdown('**Suggested next steps:** Review target occupation requirements; develop the priority skills above through a relevant project or course; confirm Australian qualifications and local vacancies independently. These are development suggestions, not verified training-course recommendations.'))
    if save:
        top.to_csv(RESULTS/'latest_skill_recommendations.csv',index=False)
        if market_rank is not None:market_rank.to_csv(RESULTS/'latest_market_recommendations.csv',index=False)
        gaps.to_csv(RESULTS/'latest_skill_differences.csv',index=False)
        print('Saved the latest analysis in results/.')
    return top,market_rank

# Friendly skill labels from Phase 1 dictionary, where available.
labels={}
if (PROCESSED/'skill_dictionary.csv').exists():
    d=pd.read_csv(PROCESSED/'skill_dictionary.csv',dtype=str)
    k=next((x for x in ('element_id','skill_code','skill_id') if x in d.columns),None)
    v=next((x for x in ('skill_name','element_name','name') if x in d.columns),None)
    if k and v:labels=dict(zip(d[k],d[v]))
try:
    import ipywidgets as widgets
    options=sorted([(str(lookup.loc[c,'occupation_title'])+' | '+c,c) for c in eligible],key=lambda x:x[0].lower())
    search=widgets.Text(description='Search:',placeholder='Search all occupations...',layout=widgets.Layout(width='75%'))
    selector=widgets.Dropdown(options=options,description='Your occupation:',layout=widgets.Layout(width='85%'),style={'description_width':'initial'})
    count=widgets.HTML(value=f'<b>{len(options)} occupations available</b>')
    region_dropdown=widgets.Dropdown(options=['Victoria']+sorted(VIC.region.dropna().unique().tolist()),value='Victoria',description='Region:',style={'description_width':'initial'})
    choose_target=widgets.Dropdown(options=[],description='Compare target:',layout=widgets.Layout(width='85%'),style={'description_width':'initial'})
    sliders=widgets.VBox();current_sliders={};output=widgets.Output()
    button=widgets.Button(description='Analyse career options',button_style='success')
    def refresh(change=None):
        global current_sliders
        src=selector.value
        if src is None:return
        matches,_,_,_=demand_recommend(src)
        choose_target.options=[(row.occupation_title,row.occupation_code) for row in matches.itertuples()]
        ranked=X.loc[src].dropna().sort_values(ascending=False).head(6)
        current_sliders={key:widgets.IntSlider(value=3,min=1,max=5,description=str(labels.get(key,key)),style={'description_width':'180px'},layout=widgets.Layout(width='520px')) for key in ranked.index}
        sliders.children=tuple(current_sliders.values())
    def search_changed(change):
        term=change['new'].casefold().strip()
        subset=[item for item in options if term in item[0].casefold()]
        count.value=f'<b>{len(subset)} matching / {len(options)} total occupations</b>'
        selector.options=subset
    def clicked(_):
        with output:
            clear_output(wait=True)
            if selector.value is None:print('Search again and choose an occupation.');return
            try:show_career_report(selector.value,choose_target.value,{k:v.value for k,v in current_sliders.items()},save=True,region=region_dropdown.value)
            except Exception as e:print(type(e).__name__+': '+str(e))
    search.observe(search_changed,names='value')
    selector.observe(refresh,names='value')
    button.on_click(clicked);refresh()
    display(widgets.VBox([search,count,selector,region_dropdown,choose_target,widgets.HTML('<b>Optional personal skill confidence (1=low, 5=high):</b>'),sliders,button,output]))
except ImportError:
    print('ipywidgets is not installed. Use the manual method below.')
except Exception as exc:
    print('Widget setup failed:',repr(exc))

## 7 — Manual fallback and multi-occupation tests
This cell proves that recommendation logic works **beyond Software Developers**. Change the `keyword` below to any available occupation title if widgets are unavailable. Results will only include market evidence for reviewed mappings.

In [ ]:
# Run programmatic tests across several distinct occupations.
test_codes=list(eligible[::max(1,len(eligible)//30)][:30])
for src in test_codes:
    rec,market_rec,message,all_results=demand_recommend(src)
    assert len(rec)==5 and src not in set(rec.occupation_code)
    assert rec.similarity.between(0,1.000001).all()
    assert rec.similarity.is_monotonic_decreasing
    assert not rec.occupation_code.duplicated().any()
    if market_rec is not None:
        assert market_rec.growth_pct_2025_2035.notna().all()
        assert market_rec.combined_score.is_monotonic_decreasing
print('Passed 30-or-fewer occupation invariants:',len(test_codes))
print('Market coverage:',len(M),'reviewed mappings across',len(eligible),'eligible O*NET occupations.')
# Manual use: edit the keyword and optionally call show_career_report manually.
keyword='retail'
found=search_jobs(keyword)
print('Manual search results:');display(found)
if len(found):
    print('Manual run example: show_career_report(found.index[0])')

## Responsible interpretation
- **All 910 occupations:** skills recommendations plus three automatic ANZSCO matching candidates; proposals are not verified crosswalks.
- **VSA data:** 2025–2035 forecasts at ANZSCO4 group level, including 2025–2028 new-worker expectations (growth plus replacement).
- **Market-ranked recommendations:** only from the 50 closest skills neighbours, with a reviewed mapping or title-similarity threshold; candidates marked `VSA_GROUP_UNVERIFIED_MAPPING` are exploratory, **not reliable occupational forecasts**.
- A high skill similarity does not establish job eligibility, licensing, credential equivalence, or actual openings. Missing market evidence should not be interpreted as zero demand.
- The CSV `results/all_910_candidate_mappings.csv` contains proposed Australian occupation matches for review. It is deliberately *not* named `verified_mapping.csv`.


## Submission note
For an accurate demonstration, show **skills-only** top-five results (all occupations) and the separate **indicative** Victorian opportunity comparison. Do not claim 910 government-approved occupational equivalences. To establish that claim, a validated occupation concordance would be needed.
